# EEG-Auth on BED: how well does EEG login work across days? (Google Colab + Google Drive)

**Question.** A login system must recognise you days after you enrolled. This notebook measures that on **BED** (Arnau-González et al., *IEEE Internet of Things Journal* 8(15), 2021): 21 subjects, Emotiv EPOC+ (14 channels), 3 sessions one week apart, and tests ideas for closing the gap between *same-session* and *cross-session* error. It regenerates everything from your raw files.

**Why these experiments (from the literature).**
- Cross-session testing is the honest test: electrode impedance and placement change between sessions, and single-session numbers are optimistic (e.g. arXiv:2007.13517; Fallahi et al., arXiv:2501.17866 multi-session benchmark).
- Per-session alignment (Euclidean alignment, He & Wu 2018; Riemannian recentering, Zanini et al. 2018) is the standard cheap fix for session shift in EEG.
- Resting eyes-closed alpha is reported as the most stable biometric feature; averaging scores over several epochs raises accuracy (MERL 2016; score fusion in PMC5855894); pairwise/template matching suits open-set verification.
- These are hypotheses to test here, not results. Nothing is reported until you run it.

**Scenarios (EER %, lower is better, chance = 50%).**

| | Scenario | Question |
|---|---|---|
| S1 | same-session | upper bound: enrol and test within one visit |
| S2 | cross-session, 1 enrolment session | all 6 enrol → test pairs; 1-week vs 2-week gap |
| S3 | multi-session enrolment | does enrolling on 2 sessions help the 3rd? |
| S4 | enrolment size | how many clips from the new session are needed? |
| S5 | session normalisation | none / feature z-score / Euclidean alignment / Riemannian recentering |
| S6 | resting condition | eyes-closed only vs eyes-open only vs both |
| S7 | score fusion | average k = 1, 3, 5, 10 clips per login attempt |
| S8 | feature set | all bands / alpha / theta+alpha / alpha+beta |
| S9 | open-set | impostors are people who were never enrolled |
| S10 | improvement ladder | baseline → +alignment → +2 sessions → +fusion, tested on session 3 |

Models: logistic regression on band power, Riemannian tangent space + LogReg, Riemannian **template matching** (cosine, open-set style), and the project's EEGNet-style CNN (S1, S2, S3, S10). Hyper-parameters are tuned on a time-block validation split of session-1 *training* data only; scalers and alignments use training/unlabelled data of the relevant session only; the CNN uses 5 seeds.

**How to use**
1. *Runtime > Change runtime type > GPU* (only the CNN uses it).
2. In cell 4 set `DRIVE_BED_DIR` to the Drive folder with the BED `RAW_PARSED` `.mat` files (or a zip). If wrong, the notebook searches your Drive.
3. *Runtime > Run all*; authorise Drive in cell 3. Cell 8 prints the real file structure: check the resting-state labels were recognised.
4. Results land in `MyDrive/eeg_auth_bed/results_<RUN_TAG>/` (tables, csv, json, figures). Each run is also logged in `MyDrive/eeg_auth_bed/runs/runs.jsonl`, and earlier runs are never overwritten: change `RUN_TAG` for a new experiment and the history stays.

BED is restricted: keep the Drive folder private; never commit or share the data. Processed clips and results are saved after every subject / scenario, so a disconnect loses nothing (run all again to resume).

In [ ]:
# 1. Hardware and disk
!nvidia-smi || echo "No GPU attached (CPU only: fine for LogReg/Riemannian, slower CNN)"
!df -h /content

In [ ]:
# 2. Get the project code (models, metrics, training helpers) and install what Colab lacks
import os
REPO = '/content/ECG-brain-auth'
if not os.path.exists(REPO):
    !git clone -q https://github.com/rabiaryal/ECG-brain-auth.git {REPO}
    # private repo? store a GitHub token in Colab Secrets as GITHUB_TOKEN and use:
    # from google.colab import userdata
    # !git clone -q https://{userdata.get('GITHUB_TOKEN')}@github.com/rabiaryal/ECG-brain-auth.git {REPO}
!pip install -q mne pyriemann mat73 pyyaml pandas
import sys
sys.path.insert(0, f'{REPO}/src')
print('repo ready:', os.listdir(f'{REPO}/src/eegauth'))

In [ ]:
# 3. Connect Google Drive (a pop-up asks you to authorise access)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 4. Configuration (edit here only)
N_SUBJECTS   = 21          # BED has 21 subjects
N_SEEDS      = 5           # seeds for the CNN (LogReg / Riemannian are deterministic)
TARGET_FS    = 160.0       # resample 256 Hz -> 160 Hz
BAND         = (1.0, 40.0) # band-pass, Hz
CLIP_S       = 2.0         # clip length, seconds
MAX_SECONDS  = None        # optional cap per resting segment (seconds); None = use all
TRAIN_FRAC   = 0.7         # first 70% (by time) of every session-1 segment = training
VAL_FRAC     = 0.25        # last 25% (by time) of the training part = tuning validation
MAX_EPOCHS   = 60          # CNN max epochs (early stopping on a split of the training clips)
RUN_TAG      = 'v1'        # name of this run; results go to results_<RUN_TAG>/ (change it to keep older runs)
SCENARIOS    = ['S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10']   # drop some to run faster
USE_CNN      = True        # CNN adds the most run time (S1, S2, S3, S10 only)
CNN_SCENARIOS = ['S1', 'S2', 'S3', 'S10']
RESUME       = True        # reuse finished scenarios of the same RUN_TAG after a disconnect
# Which events are resting state? Regexes on the lower-cased event label. Check them against the
# labels printed by cell 8 and edit if they don't match.
REST_RULES   = {'eyes_closed': r'clos', 'eyes_open': r'open'}
LABEL_FIELD  = None        # name of the ADDITIONAL_INFO field holding the stimulus name; None = auto
# Google Drive locations (MyDrive = your Drive root)
DRIVE_BED_DIR = '/content/drive/MyDrive/BED'            # <-- folder in YOUR Drive that holds the BED files; edit this
WORK          = '/content/drive/MyDrive/eeg_auth_bed'   # processed data + results are saved here (Drive, resumable)
print('Drive BED folder:', DRIVE_BED_DIR)
print('Work/results folder:', WORK)

In [ ]:
%%writefile /content/ECG-brain-auth/src/eegauth/bed.py
"""BED (Biometric EEG Dataset) support for EEG-Auth: parsing, rest clips, protocols.

BED: Arnau-Gonzalez, Katsigiannis, Arevalillo-Herraez, Ramzan, IEEE Internet of
Things Journal 8(15), 2021 (Zenodo 4309471). RESTRICTED data: keep it out of git.

Documented format of RAW_PARSED/sN_sM.mat:
    recording : (time @ 256 Hz, 17) COUNTER INTERPOLATED F3 FC5 AF3 F7 T7 P7 O1 O2 P8 T8 F8 AF4 FC6 F4 UNIX_TIMESTAMP
    events    : cell (n, 3) = START_UNIX, END_UNIX, ADDITIONAL_INFO (struct)
The stimulus labels inside ADDITIONAL_INFO are not documented: they are
discovered at run time and matched with configurable regex rules.

Protocol: train = first `train_frac` (by time) of every session-1 segment;
same-session test = the later part of those segments (after a gap);
1-week test = session 2; 2-week test = session 3. One training set for all
three columns keeps them comparable.
"""
from __future__ import annotations

import json
import re
import time
from math import gcd
from pathlib import Path
from typing import Any, Callable, Dict, List, Sequence, Tuple

import numpy as np

EMOTIV_COLUMNS = ["COUNTER", "INTERPOLATED", "F3", "FC5", "AF3", "F7", "T7", "P7", "O1",
                  "O2", "P8", "T8", "F8", "AF4", "FC6", "F4", "UNIX_TIMESTAMP"]
BED_CHANNELS = ["AF3", "F7", "F3", "FC5", "T7", "P7", "O1", "O2", "P8", "T8",
                "FC6", "F4", "F8", "AF4"]
SOURCE_FS = 256
TASKS = ["eyes_closed", "eyes_open"]
DEFAULT_RULES = {"eyes_closed": r"clos", "eyes_open": r"open"}
LABEL_KEYS = ("stimuli", "stimulus", "stim", "type", "name", "label", "task",
              "condition", "event", "class")


# ------------------------------ reading ------------------------------------ #
def load_bed_mat(path):
    from scipy.io import loadmat

    try:
        m = loadmat(str(path), squeeze_me=True, struct_as_record=False)
    except NotImplementedError:  # MATLAB v7.3 (HDF5)
        import mat73

        m = mat73.loadmat(str(path))
    if "recording" not in m or "events" not in m:
        raise KeyError(f"{path}: expected 'recording' and 'events', found "
                       f"{[k for k in m if not k.startswith('__')]}")
    return np.asarray(m["recording"], dtype=np.float64), m["events"]


def _plain(x, depth=0):
    if depth > 6:
        return "..."
    if hasattr(x, "_fieldnames"):
        return {k: _plain(getattr(x, k), depth + 1) for k in x._fieldnames}
    if isinstance(x, dict):
        return {str(k): _plain(v, depth + 1) for k, v in x.items()}
    if isinstance(x, bytes):
        return x.decode("utf-8", "replace")
    if isinstance(x, np.ndarray):
        if x.dtype == object:
            return [_plain(i, depth + 1) for i in x.ravel().tolist()[:20]]
        return x.tolist() if x.size <= 10 else f"<array {x.shape} {x.dtype}>"
    if isinstance(x, np.generic):
        return x.item()
    if isinstance(x, (list, tuple)):
        return [_plain(i, depth + 1) for i in list(x)[:20]]
    return x


def parse_events(events) -> List[Dict[str, Any]]:
    if isinstance(events, np.ndarray) and events.ndim == 1 and events.shape[0] == 3:
        rows = [list(events)]
    elif isinstance(events, np.ndarray):
        rows = [list(r) for r in np.atleast_2d(events)]
    else:
        rows = [list(r) for r in events]
    return [{"start": float(np.asarray(r[0]).ravel()[0]), "end": float(np.asarray(r[1]).ravel()[0]),
             "info": _plain(r[2]) if len(r) > 2 else {}} for r in rows]


def event_label(info, label_field=None) -> str:
    if isinstance(info, dict):
        if label_field and label_field in info:
            return str(info[label_field]).lower()
        lower = {k.lower(): v for k, v in info.items()}
        for k in LABEL_KEYS:
            if isinstance(lower.get(k), str):
                return lower[k].lower()
        strs = [v for v in info.values() if isinstance(v, str)]
        if strs:
            return " ".join(strs).lower()
    return str(info).lower()[:200]


def classify_rest(label, rules=None):
    rules = rules or DEFAULT_RULES
    for task in TASKS:  # closed first
        if re.search(rules[task], label):
            return task
    return None


def inspect_mat(path, max_events=5) -> Dict[str, Any]:
    rec, ev = load_bed_mat(path)
    events = parse_events(ev)
    ts = rec[:, -1]
    labels: Dict[str, int] = {}
    for e in events:
        lab = event_label(e["info"])
        labels[lab] = labels.get(lab, 0) + 1
    d = np.median(np.diff(ts)) if rec.shape[0] > 2 else 0
    return {"recording_shape": rec.shape, "assumed_columns": EMOTIV_COLUMNS,
            "assumed_sampling_rate_hz": SOURCE_FS,
            "measured_rate_hz": float(1.0 / d) if d > 0 else None,
            "timestamp_range": (float(ts[0]), float(ts[-1])), "n_events": len(events),
            "first_events": events[:max_events], "label_counts": labels,
            "info_keys": sorted({k for e in events if isinstance(e["info"], dict) for k in e["info"]}),
            "nan_in_recording": int(np.isnan(rec).sum())}


# ---------------------------- preprocessing -------------------------------- #
def cut_clips(data, clip_samples):
    n = data.shape[1] // clip_samples
    if n == 0:
        return np.empty((0, data.shape[0], clip_samples), dtype=np.float32)
    x = data[:, : n * clip_samples].reshape(data.shape[0], n, clip_samples)
    return x.transpose(1, 0, 2).astype(np.float32)


def extract_rest_clips(rec, events_raw, *, rules=None, label_field=None, target_fs=160.0,
                       source_fs=SOURCE_FS, l_freq=1.0, h_freq=40.0, clip_s=2.0,
                       trim_s=1.0, max_seconds=None) -> Dict[str, Any]:
    """Filter whole recording (1-40 Hz), cut rest events, resample, cut 2 s clips."""
    import mne
    from scipy.signal import resample_poly

    cols = [EMOTIV_COLUMNS.index(c) for c in BED_CHANNELS]
    data = rec[:, cols].T
    if not np.isfinite(data).all():
        raise ValueError("recording contains NaN/Inf")
    ts = rec[:, EMOTIV_COLUMNS.index("UNIX_TIMESTAMP")]
    data = mne.filter.filter_data(data, source_fs, l_freq, h_freq, verbose="ERROR")
    up, down = int(target_fs), int(source_fs)
    g = gcd(up, down)
    up, down = up // g, down // g
    clip_samples = int(round(clip_s * target_fs))
    events = parse_events(events_raw)
    scale = 1e-3 if events and events[0]["start"] > 1e11 and ts[-1] < 1e11 else 1.0
    xs, tasks, segs, poss, n_rest = [], [], [], [], 0
    for k, e in enumerate(events):
        task = classify_rest(event_label(e["info"], label_field), rules)
        if task is None:
            continue
        n_rest += 1
        i0 = int(np.searchsorted(ts, e["start"] * scale)) + int(trim_s * source_fs)
        i1 = int(np.searchsorted(ts, e["end"] * scale)) - int(trim_s * source_fs)
        if max_seconds is not None:
            i1 = min(i1, i0 + int(max_seconds * source_fs))
        if i1 - i0 < int(clip_s * source_fs):
            continue
        clips = cut_clips(resample_poly(data[:, i0:i1], up, down, axis=1), clip_samples)
        if len(clips) == 0:
            continue
        xs.append(clips)
        tasks += [TASKS.index(task)] * len(clips)
        segs += [k] * len(clips)
        poss += list(range(len(clips)))
    X = np.concatenate(xs) if xs else np.empty((0, len(BED_CHANNELS), clip_samples), np.float32)
    return {"X": X, "task": np.asarray(tasks, np.int8), "seg": np.asarray(segs, np.int32),
            "pos": np.asarray(poss, np.int32),
            "diag": {"n_events": len(events), "n_rest_events": n_rest, "n_clips": int(len(X))}}


def combine_arrays(parts):
    return {k: np.concatenate([p[k] for p in parts])
            for k in ("X", "subject", "session", "task", "seg", "pos")}


def find_mats(root) -> Dict[Tuple[int, int], Path]:
    """Map (subject, session) -> path for files named sN_sM.mat under root."""
    found = {}
    for p in sorted(Path(root).rglob("*.mat")):
        m = re.fullmatch(r"s(\d+)_s(\d+)\.mat", p.name, flags=re.I)
        if m:
            found.setdefault((int(m.group(1)), int(m.group(2))), p)
    return found


def prepare_dataset(raw_dir, out_dir, *, mats=None, n_subjects=21, rules=None, label_field=None,
                    target_fs=160.0, l_freq=1.0, h_freq=40.0, clip_s=2.0, max_seconds=None,
                    log=print) -> Dict[str, Any]:
    """Per subject (resumable): .mat -> rest clips -> out_dir/processed/sNN.npz; then combine.

    ``mats`` = optional {(subject, session): path}; otherwise files are searched under raw_dir.
    """
    rules = rules or DEFAULT_RULES
    out_dir = Path(out_dir)
    proc = out_dir / "processed"
    proc.mkdir(parents=True, exist_ok=True)
    mats = mats if mats is not None else find_mats(raw_dir)
    skipped, per_rec = [], {}
    for sub in range(1, n_subjects + 1):
        target = proc / f"s{sub:02d}.npz"
        if target.exists():
            log(f"s{sub}: already processed, skipping")
            continue
        parts, t0 = [], time.time()
        for ses in (1, 2, 3):
            path = mats.get((sub, ses))
            if path is None:
                skipped.append({"subject": sub, "session": ses, "reason": "file not found"})
                log(f"s{sub} session {ses}: file not found")
                continue
            try:
                rec, ev = load_bed_mat(path)
                r = extract_rest_clips(rec, ev, rules=rules, label_field=label_field,
                                       target_fs=target_fs, l_freq=l_freq, h_freq=h_freq,
                                       clip_s=clip_s, max_seconds=max_seconds)
            except Exception as exc:  # log and continue
                skipped.append({"subject": sub, "session": ses, "reason": repr(exc)})
                log(f"s{sub} session {ses}: FAILED {exc!r}")
                continue
            if r["diag"]["n_clips"] == 0:
                skipped.append({"subject": sub, "session": ses, "reason": f"no resting clips {r['diag']}"})
                log(f"s{sub} session {ses}: no resting clips {r['diag']}")
                continue
            n = len(r["X"])
            parts.append({"X": r["X"].astype(np.float16), "subject": np.full(n, sub, np.int16),
                          "session": np.full(n, ses, np.int8), "task": r["task"],
                          "seg": r["seg"], "pos": r["pos"]})
            per_rec[f"s{sub}_s{ses}"] = r["diag"]
            log(f"s{sub} session {ses}: {r['diag']}")
        if parts:
            np.savez_compressed(target, **combine_arrays(parts))
            log(f"s{sub} saved ({time.time() - t0:.1f}s)")
    files = sorted(proc.glob("s*.npz"))
    if not files:
        raise RuntimeError("no subject produced resting clips: check REST_RULES / LABEL_FIELD "
                           "against the labels printed by the inspect cell")
    data = combine_arrays([dict(np.load(f)) for f in files])
    np.savez_compressed(out_dir / "bed_subset.npz", **data)
    meta = {"dataset": "BED (Arnau-Gonzalez et al., IEEE IoT J. 2021) - RESTRICTED, do not publish",
            "channels": BED_CHANNELS, "source_fs_hz": SOURCE_FS, "target_fs_hz": target_fs,
            "band_hz": [l_freq, h_freq], "clip_s": clip_s, "dtype": "float16", "tasks": TASKS,
            "rules": rules, "label_field": label_field, "n_clips": int(len(data["X"])),
            "subjects": sorted({int(s) for s in data["subject"]}),
            "clips_per_session": {str(s): int((data["session"] == s).sum()) for s in (1, 2, 3)},
            "clips_per_task": {t: int((data["task"] == i).sum()) for i, t in enumerate(TASKS)},
            "skipped": skipped, "per_recording": per_rec}
    (out_dir / "bed_subset_meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
    return meta


# ------------------------------- splits ------------------------------------ #
def time_block_split(meta, idx, frac, gap=1):
    """Within each (subject, session, task, segment): first `frac` -> head, skip `gap`, rest -> tail."""
    groups: Dict[Tuple[int, int, int, int], List[int]] = {}
    for i in idx:
        key = (int(meta["subject"][i]), int(meta["session"][i]), int(meta["task"][i]), int(meta["seg"][i]))
        groups.setdefault(key, []).append(int(i))
    head, tail = [], []
    for members in groups.values():
        members.sort(key=lambda i: int(meta["pos"][i]))
        n = len(members)
        cut = min(max(int(round(n * frac)), 1), n)
        head += members[:cut]
        tail += members[cut + gap:]
    return np.asarray(sorted(head), dtype=int), np.asarray(sorted(tail), dtype=int)


def build_splits(meta, train_frac=0.7, val_frac=0.25, gap=1):
    sess = meta["session"]
    train, same = time_block_split(meta, np.flatnonzero(sess == 1), train_frac, gap)
    tr_sub, val = time_block_split(meta, train, 1.0 - val_frac, gap)
    return {"train": train, "same_session": same, "tune_train": tr_sub, "tune_val": val,
            "week1": np.flatnonzero(sess == 2), "week2": np.flatnonzero(sess == 3)}


def eligible_subjects(meta, splits):
    sets = [set(meta["subject"][splits[k]].tolist()) for k in ("train", "week1", "week2")]
    return np.asarray(sorted(set.intersection(*sets)), dtype=int)


# ------------------------------- models ------------------------------------ #
class _Base:
    name = ""
    grid: List[Dict[str, Any]] = []
    deterministic = True


class LogRegBandPower(_Base):
    name = "LogReg (band power)"
    grid = [{"C": c} for c in (0.01, 0.1, 1.0, 10.0)]

    def fit(self, X, Xf, y, idx, params, seed, n_classes):
        from eegauth.train import fit_baseline, score_matrix

        pipe = fit_baseline(Xf[idx], y[idx], kind="logreg", C=params["C"], random_state=seed)
        return lambda j: score_matrix(pipe, Xf[j], n_classes)


class RiemannianTangent(_Base):
    name = "Riemannian (tangent space + LogReg)"
    grid = [{"C": c} for c in (0.01, 0.1, 1.0)]

    def fit(self, X, Xf, y, idx, params, seed, n_classes):
        from pyriemann.estimation import Covariances
        from pyriemann.tangentspace import TangentSpace
        from sklearn.linear_model import LogisticRegression
        from sklearn.pipeline import Pipeline
        from sklearn.preprocessing import StandardScaler

        from eegauth.train import PerChannelNormalizer, score_matrix

        norm = PerChannelNormalizer().fit(X[idx])  # train stats only
        pipe = Pipeline([("cov", Covariances(estimator="oas")),
                         ("ts", TangentSpace(metric="riemann")), ("sc", StandardScaler()),
                         ("clf", LogisticRegression(C=params["C"], max_iter=2000, random_state=seed))])
        pipe.fit(norm.transform(X[idx]).astype(np.float64), y[idx])
        return lambda j: score_matrix(pipe, norm.transform(X[j]).astype(np.float64), n_classes)


class EEGNetStyleCNN(_Base):
    """The repo's compact EEGNet-style CNN (EEGCNNSmall), not the original EEGNet."""
    name = "EEGNet-style CNN"
    grid = [{"n_filters": f, "dropout": d} for f in (16, 32) for d in (0.3, 0.5)]
    deterministic = False

    def __init__(self, max_epochs=60, patience=8, lr=1e-3, batch_size=64):
        self.max_epochs, self.patience, self.lr, self.batch_size = max_epochs, patience, lr, batch_size

    def fit(self, X, Xf, y, idx, params, seed, n_classes):
        import torch

        from eegauth.models import EEGCNNSmall, get_device
        from eegauth.train import PerChannelNormalizer, cnn_softmax_scores, train_cnn

        device = get_device()
        torch.manual_seed(seed)
        norm = PerChannelNormalizer().fit(X[idx])
        Xtr = norm.transform(X[idx])
        model = EEGCNNSmall(n_channels=Xtr.shape[1], n_samples=Xtr.shape[2], n_classes=n_classes,
                            n_filters=params["n_filters"], dropout=params["dropout"]).to(device)
        train_cnn(model, Xtr, y[idx], lr=self.lr, batch_size=self.batch_size,
                  max_epochs=self.max_epochs, patience=self.patience, val_fraction=0.1,
                  device=device, seed=seed, verbose=False)
        model.eval()
        return lambda j: cnn_softmax_scores(model, norm.transform(X[j]), device=device)


# ----------------------------- evaluation ---------------------------------- #
def _eval(scores, y):
    from eegauth.metrics import verification_metrics
    from eegauth.train import genuine_impostor_scores

    gen, imp = genuine_impostor_scores(scores, y)
    m = verification_metrics(gen, imp)
    return {"identification_accuracy": float(np.mean(scores.argmax(1) == y)), "eer": m.eer,
            "auc": m.auc, "far_at_eer": m.far_at_eer, "frr_at_eer": m.frr_at_eer, "n_clips": int(len(y))}


def run_protocols(data, models, *, seeds=(0, 1, 2, 3, 4), train_frac=0.7, val_frac=0.25, gap=1,
                  sfreq=160.0, log=print) -> Dict[str, Any]:
    """Tune on a validation split of the training data only, then evaluate 3 protocols."""
    from eegauth.features import log_band_power_features

    meta = {k: data[k] for k in ("subject", "session", "task", "seg", "pos")}
    splits = build_splits(meta, train_frac, val_frac, gap)
    subs = eligible_subjects(meta, splits)
    if len(subs) < 2:
        raise ValueError(f"need >= 2 subjects present in sessions 1, 2 and 3; got {len(subs)}")
    keep = np.isin(meta["subject"], subs)
    remap = {int(s): i for i, s in enumerate(subs)}
    y_all = np.array([remap.get(int(s), -1) for s in meta["subject"]])
    splits = {k: v[keep[v]] for k, v in splits.items()}
    n_classes = len(subs)
    X = data["X"].astype(np.float32)
    Xf = log_band_power_features(X, sfreq=sfreq)
    log(f"subjects used: {n_classes}; clips train={len(splits['train'])}, "
        f"same-session={len(splits['same_session'])}, 1-week={len(splits['week1'])}, "
        f"2-week={len(splits['week2'])}")
    out: Dict[str, Any] = {"n_subjects": n_classes, "subjects": subs.tolist(),
                           "clips": {k: int(len(v)) for k, v in splits.items()},
                           "seeds": list(seeds), "models": {}}
    for model in models:
        best, best_key, tuning = None, (np.inf, 0.0), []
        for params in model.grid:  # tuning: train-only split, first seed
            scorer = model.fit(X, Xf, y_all, splits["tune_train"], params, seeds[0], n_classes)
            r = _eval(scorer(splits["tune_val"]), y_all[splits["tune_val"]])
            tuning.append({"params": params, "val_eer": r["eer"], "val_acc": r["identification_accuracy"]})
            key = (r["eer"], -r["identification_accuracy"])
            if key < best_key:
                best, best_key = params, key
        log(f"[{model.name}] best params {best} (val EER {best_key[0]:.4f})")
        per_seed = []
        for sd in (list(seeds) if not model.deterministic else [seeds[0]]):
            scorer = model.fit(X, Xf, y_all, splits["train"], best, sd, n_classes)
            res = {p: _eval(scorer(splits[p]), y_all[splits[p]]) for p in ("same_session", "week1", "week2")}
            per_seed.append(res)
            log(f"[{model.name}] seed {sd}: " + " | ".join(f"{p} EER {res[p]['eer']:.4f}" for p in res))
        agg: Dict[str, Any] = {}
        for proto in ("same_session", "week1", "week2"):
            for metric in ("eer", "identification_accuracy"):
                v = np.array([s[proto][metric] for s in per_seed])
                agg.setdefault(proto, {})[metric] = {"mean": float(v.mean()), "std": float(v.std()),
                                                     "n_runs": int(len(v))}
        out["models"][model.name] = {"best_params": best, "tuning": tuning,
                                     "deterministic": model.deterministic,
                                     "per_seed": per_seed, "summary": agg}
    return out


def results_table(results, metric="eer") -> str:
    fmt = lambda s: f"{100 * s['mean']:.2f} ± {100 * s['std']:.2f}%"
    lines = ["| Model | Same-session | 1-week | 2-week |", "|---|---|---|---|"]
    for name, r in results["models"].items():
        s = r["summary"]
        lines.append(f"| {name} | " + " | ".join(fmt(s[p][metric]) for p in ("same_session", "week1", "week2")) + " |")
    return "\n".join(lines)


def save_report(results, out_dir) -> str:
    """Write bed_results.json, bed_table.md and bed_eer.png; return the markdown."""
    import matplotlib.pyplot as plt

    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    (out / "bed_results.json").write_text(json.dumps(results, indent=2), encoding="utf-8")
    md = (f"## EER (mean ± std over seeds)\n\n{results_table(results, 'eer')}\n\n"
          f"## Identification accuracy\n\n{results_table(results, 'identification_accuracy')}\n\n"
          f"Subjects: {results['n_subjects']}; clips: {results['clips']}.\n"
          "Deterministic models (LogReg, Riemannian) run once; their std is 0 by construction.\n")
    (out / "bed_table.md").write_text(md, encoding="utf-8")
    protos = [("same_session", "same-session"), ("week1", "1 week"), ("week2", "2 weeks")]
    fig, ax = plt.subplots(figsize=(7, 4))
    k = max(len(results["models"]), 1)
    w = 0.8 / k
    for i, (name, r) in enumerate(results["models"].items()):
        ax.bar(np.arange(3) + i * w, [100 * r["summary"][p]["eer"]["mean"] for p, _ in protos], w,
               yerr=[100 * r["summary"][p]["eer"]["std"] for p, _ in protos], capsize=3, label=name)
    ax.set_xticks(np.arange(3) + w * (k - 1) / 2)
    ax.set_xticklabels([l for _, l in protos])
    ax.set_ylabel("EER (%)")
    ax.set_title(f"BED: EER by time gap to training session ({results['n_subjects']} subjects)")
    ax.legend(fontsize=8)
    fig.savefig(out / "bed_eer.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    return md


# =========================================================================== #
#                      SCENARIO ENGINE (cross-session study)                  #
# =========================================================================== #
# Every scenario answers one question about EEG login across days. All of them
# share one evaluation core (`evaluate`) so numbers are comparable:
#   * models are fitted on the "train" clips only (scalers / whiteners too),
#   * per-session normalisation uses UNLABELLED clips of that session only,
#   * EER uses zero-effort impostors (the other enrolled users).
BANDS = {"all": {"theta": (4.0, 8.0), "alpha": (8.0, 13.0), "beta": (13.0, 30.0), "gamma": (30.0, 40.0)},
         "alpha": {"alpha": (8.0, 13.0)},
         "theta_alpha": {"theta": (4.0, 8.0), "alpha": (8.0, 13.0)},
         "alpha_beta": {"alpha": (8.0, 13.0), "beta": (13.0, 30.0)}}
NORM_MODES = ("none", "fz", "ea", "riem")
NORM_HELP = {"none": "no session normalisation",
             "fz": "z-score band-power features per session",
             "ea": "Euclidean alignment (whiten each session by its mean covariance) + feature z-score",
             "riem": "Riemannian recentering (whiten by the Riemannian mean covariance) + feature z-score"}


class TemplateCosine(_Base):
    """Open-set style verifier: per-user template = mean tangent-space vector, score = cosine similarity.

    No classifier is trained against other people, so it can score users it was not trained against."""
    name = "Riemannian template (cosine)"
    grid = [{}]

    def fit(self, X, Xf, y, idx, params, seed, n_classes):
        from pyriemann.estimation import Covariances
        from pyriemann.tangentspace import TangentSpace
        from sklearn.preprocessing import StandardScaler

        from eegauth.train import PerChannelNormalizer

        norm = PerChannelNormalizer().fit(X[idx])
        cov = Covariances(estimator="oas")
        ts = TangentSpace(metric="riemann")
        sc = StandardScaler()
        Z = sc.fit_transform(ts.fit_transform(cov.transform(norm.transform(X[idx]).astype(np.float64))))
        T = np.zeros((n_classes, Z.shape[1]))
        for c in np.unique(y[idx]):
            T[c] = Z[y[idx] == c].mean(0)
        T /= np.linalg.norm(T, axis=1, keepdims=True) + 1e-12

        def scorer(j):
            Q = sc.transform(ts.transform(cov.transform(norm.transform(X[j]).astype(np.float64))))
            Q /= np.linalg.norm(Q, axis=1, keepdims=True) + 1e-12
            return Q @ T.T
        return scorer


# probability-producing models are fused in log space, cosine scores directly
LogRegBandPower.prob = True
RiemannianTangent.prob = True
EEGNetStyleCNN.prob = True
TemplateCosine.prob = False


class Ctx:
    """Clips of the subjects that have all 3 sessions + cached features."""

    def __init__(self, data, sfreq=160.0, train_frac=0.7, val_frac=0.25, log=print):
        subj = data["subject"].astype(int)
        sess = data["session"].astype(int)
        ok = [s for s in np.unique(subj) if all(((subj == s) & (sess == k)).any() for k in (1, 2, 3))]
        if len(ok) < 3:
            raise ValueError(f"need >= 3 subjects with sessions 1, 2 and 3; got {len(ok)}")
        keep = np.isin(subj, ok)
        self.subjects = np.asarray(ok)
        remap = {int(s): i for i, s in enumerate(ok)}
        self.subject = subj[keep]
        self.y = np.array([remap[int(s)] for s in self.subject])
        self.session = sess[keep]
        self.task = data["task"][keep].astype(int)
        self.seg = data["seg"][keep].astype(int)
        self.pos = data["pos"][keep].astype(int)
        self.X = data["X"][keep].astype(np.float32)
        self.n = len(ok)
        self.sfreq = sfreq
        self.meta = {"subject": self.subject, "session": self.session, "task": self.task,
                     "seg": self.seg, "pos": self.pos}
        self.splits = build_splits(self.meta, train_frac, val_frac)
        self._feats, self._prep = {}, {}
        log(f"{self.n} subjects; clips per session: "
            + ", ".join(f"s{k}={int((self.session == k).sum())}" for k in (1, 2, 3)))

    def idx(self, session=None, task=None):
        m = np.ones(len(self.y), bool)
        if session is not None:
            m &= np.isin(self.session, np.atleast_1d(session))
        if task is not None:
            m &= self.task == task
        return np.flatnonzero(m)

    def first_n_per_subject(self, idx, n):
        if n <= 0:
            return idx[:0]
        order = np.lexsort((self.pos[idx], self.seg[idx], self.task[idx], self.subject[idx]))
        out, seen = [], {}
        for i in idx[order]:
            s = int(self.subject[i])
            if seen.get(s, 0) < n:
                out.append(i)
                seen[s] = seen.get(s, 0) + 1
        return np.asarray(sorted(out), dtype=int)

    def feats(self, bands="all"):
        if bands not in self._feats:
            from eegauth.features import log_band_power_features
            self._feats[bands] = log_band_power_features(self.X, sfreq=self.sfreq, bands=BANDS[bands])
        return self._feats[bands]


def _whitener(Xref, kind):
    C = np.einsum("nct,ndt->ncd", Xref.astype(np.float64), Xref.astype(np.float64)) / Xref.shape[2]
    eye = np.eye(C.shape[1])
    C = C + 1e-6 * np.trace(C, axis1=1, axis2=2)[:, None, None] / C.shape[1] * eye
    if kind == "ea":
        R = C.mean(0)
    else:
        from pyriemann.utils.mean import mean_riemann
        R = mean_riemann(C)
    R = R + 1e-6 * np.trace(R) / len(R) * eye
    w, V = np.linalg.eigh(R)
    return (V / np.sqrt(w)) @ V.T


def prepare_inputs(ctx, mode, used, bands="all"):
    """(X, Xf) for a normalisation mode. Statistics of each session come from the clips of
    that session that the scenario actually uses (`used`), never from labels."""
    key = (mode, bands, hash(np.asarray(used).tobytes()))
    if key in ctx._prep:
        return ctx._prep[key]
    Xf = ctx.feats(bands)
    if mode == "none":
        out = (ctx.X, Xf)
    else:
        Xf2 = Xf.copy()
        X2 = ctx.X.copy() if mode in ("ea", "riem") else ctx.X
        for s in (1, 2, 3):
            m = ctx.session == s
            ref = used[ctx.session[used] == s]
            if len(ref) < 10:
                continue
            mu, sd = Xf[ref].mean(0), Xf[ref].std(0) + 1e-8
            Xf2[m] = (Xf[m] - mu) / sd
            if mode in ("ea", "riem"):
                W = _whitener(ctx.X[ref], mode)
                X2[m] = np.einsum("cd,ndt->nct", W, ctx.X[m].astype(np.float64)).astype(np.float32)
        out = (X2, Xf2)
    if len(ctx._prep) > 12:
        ctx._prep.clear()
    ctx._prep[key] = out
    return out


def fuse_scores(ctx, scores, idx, k, prob):
    """Average the scores of k consecutive clips of the same segment (a longer login attempt)."""
    if k <= 1:
        return scores, ctx.y[idx]
    S = np.log(np.clip(scores, 1e-9, 1.0)) if prob else scores
    groups: Dict[Tuple[int, int, int, int], List[int]] = {}
    for j, i in enumerate(idx):
        groups.setdefault((int(ctx.subject[i]), int(ctx.session[i]), int(ctx.task[i]), int(ctx.seg[i])), []).append(j)
    fs, fy = [], []
    for members in groups.values():
        members.sort(key=lambda j: int(ctx.pos[idx[j]]))
        for a in range(0, len(members) - k + 1, k):
            blk = members[a:a + k]
            fs.append(S[blk].mean(0))
            fy.append(ctx.y[idx[blk[0]]])
    if not fs:
        return scores[:0], ctx.y[idx][:0]
    return np.asarray(fs), np.asarray(fy)


def evaluate(ctx, models, train, tests, params, seeds, *, mode="none", bands="all", fuse=(1,),
             scenario="", setting="", group="", log=print):
    """Fit every model on `train`, score each named test set; returns result rows."""
    train = np.asarray(train)
    if len(np.unique(ctx.y[train])) < ctx.n:
        log(f"  [skip] {scenario} | {setting}: some users have no training clips")
        return []
    used = np.unique(np.concatenate([train] + [np.asarray(v) for v in tests.values()]))
    X, Xf = prepare_inputs(ctx, mode, used, bands)
    rows = []
    for model in models:
        runs = []
        for sd in (list(seeds) if not model.deterministic else [seeds[0]]):
            scorer = model.fit(X, Xf, ctx.y, train, params.get(model.name, {}), sd, ctx.n)
            res = {}
            for tname, tidx in tests.items():
                sc = scorer(np.asarray(tidx))
                for k in fuse:
                    fs, fy = fuse_scores(ctx, sc, np.asarray(tidx), k, model.prob)
                    res[(tname, k)] = _eval(fs, fy) if len(fy) else None
            runs.append(res)
        for key in runs[0]:
            vals = [r[key] for r in runs if r[key] is not None]
            if not vals:
                continue
            e = np.array([v["eer"] for v in vals])
            a = np.array([v["identification_accuracy"] for v in vals])
            rows.append({"scenario": scenario, "setting": setting, "group": group, "model": model.name,
                         "test": key[0], "fuse_k": int(key[1]), "norm": mode, "bands": bands,
                         "eer": float(e.mean()), "eer_std": float(e.std()),
                         "acc": float(a.mean()), "acc_std": float(a.std()),
                         "n_test": int(vals[0]["n_clips"]), "n_runs": int(len(vals))})
        log(f"  {scenario} | {setting} | {model.name}: " + " ".join(
            f"{t}{'' if k == 1 else f'(k={k})'}={100 * r['eer']:.1f}%" for (t, k), r in runs[0].items() if r))
    return rows


def tune_models(ctx, models, seed=0, log=print):
    """Pick hyper-parameters on a time-block validation split of session-1 TRAINING clips only."""
    sp = ctx.splits
    out = {}
    X, Xf = ctx.X, ctx.feats("all")
    for model in models:
        best, best_key = {}, (np.inf, 0.0)
        for p in model.grid:
            scorer = model.fit(X, Xf, ctx.y, sp["tune_train"], p, seed, ctx.n)
            r = _eval(scorer(sp["tune_val"]), ctx.y[sp["tune_val"]])
            key = (r["eer"], -r["identification_accuracy"])
            if key < best_key:
                best, best_key = p, key
        out[model.name] = best
        log(f"  tuned {model.name}: {best} (val EER {100 * best_key[0]:.2f}%)")
    return out


# ------------------------------- scenarios --------------------------------- #
def sc_same_session(ctx, M, P, seeds, log):
    sp = ctx.splits
    return evaluate(ctx, M, sp["train"], {"same_session": sp["same_session"]}, P, seeds,
                    scenario="S1 same-session", setting="enrol s1 (first 70%) -> later part of s1",
                    group="same session", log=log)


def sc_cross_session(ctx, M, P, seeds, log):
    rows = []
    for a in (1, 2, 3):
        for b in (1, 2, 3):
            if a != b:
                gap = abs(a - b)
                rows += evaluate(ctx, M, ctx.idx(session=a), {f"s{b}": ctx.idx(session=b)}, P, seeds,
                                 scenario="S2 cross-session (1 enrol session)", setting=f"s{a} -> s{b}",
                                 group=f"{gap}-week gap", log=log)
    return rows


def sc_multi_session(ctx, M, P, seeds, log):
    rows = []
    for t in (1, 2, 3):
        enrol = [s for s in (1, 2, 3) if s != t]
        rows += evaluate(ctx, M, ctx.idx(session=enrol), {f"s{t}": ctx.idx(session=t)}, P, seeds,
                         scenario="S3 multi-session enrolment (2 enrol sessions)",
                         setting=f"s{enrol[0]}+s{enrol[1]} -> s{t}", group=f"test s{t}", log=log)
    return rows


def sc_enrol_size(ctx, M, P, seeds, log, sizes=(0, 5, 10, 20, 40, 80)):
    rows = []
    s1, s2, s3 = ctx.idx(session=1), ctx.idx(session=2), ctx.idx(session=3)
    for n in sizes:
        train = np.concatenate([s1, ctx.first_n_per_subject(s2, n)])
        rows += evaluate(ctx, M, train, {"s3": s3}, P, seeds, scenario="S4 enrolment size",
                         setting=f"s1 + {n} clips/user of s2 -> s3", group=str(n), log=log)
    return rows


def _single_multi(ctx):
    s1, s2, s3 = ctx.idx(session=1), ctx.idx(session=2), ctx.idx(session=3)
    return {"single (s1)": (s1, {"s2": s2, "s3": s3}), "multi (s1+s2)": (np.concatenate([s1, s2]), {"s3": s3})}


def sc_normalisation(ctx, M, P, seeds, log):
    rows = []
    for mode in NORM_MODES:
        for name, (train, tests) in _single_multi(ctx).items():
            rows += evaluate(ctx, M, train, tests, P, seeds, mode=mode, scenario="S5 session normalisation",
                             setting=f"{mode}, {name}", group=mode, log=log)
    return rows


def sc_condition(ctx, M, P, seeds, log):
    rows = []
    for label, task in (("both", None), ("eyes_closed", 0), ("eyes_open", 1)):
        s = lambda k: ctx.idx(session=k, task=task)
        setups = {"single (s1)": (s(1), {"s2": s(2), "s3": s(3)}),
                  "multi (s1+s2)": (np.concatenate([s(1), s(2)]), {"s3": s(3)})}
        for name, (train, tests) in setups.items():
            if len(train) == 0 or any(len(v) == 0 for v in tests.values()):
                log(f"  [skip] condition {label}: no clips")
                continue
            rows += evaluate(ctx, M, train, tests, P, seeds, scenario="S6 resting condition",
                             setting=f"{label}, {name}", group=label, log=log)
    return rows


def sc_fusion(ctx, M, P, seeds, log, ks=(1, 3, 5, 10)):
    rows = []
    for name, (train, tests) in _single_multi(ctx).items():
        rows += evaluate(ctx, M, train, tests, P, seeds, fuse=ks, scenario="S7 multi-clip score fusion",
                         setting=name, group=name, log=log)
    return rows


def sc_bands(ctx, M, P, seeds, log):
    rows = []
    lr = [m for m in M if isinstance(m, LogRegBandPower)]
    if not lr:
        return rows
    for b in BANDS:
        for name, (train, tests) in _single_multi(ctx).items():
            rows += evaluate(ctx, lr, train, tests, P, seeds, bands=b, scenario="S8 band-power feature set",
                             setting=f"{b}, {name}", group=b, log=log)
    return rows


def sc_openset(ctx, M, P, seeds, log, n_rep=5, enrolled_frac=2 / 3):
    """Strict open-set check: impostors are people the system has NEVER seen (not enrolled)."""
    from eegauth.metrics import verification_metrics

    tc = TemplateCosine()
    rng = np.random.default_rng(0)
    rows = []
    s1, s2, s3 = ctx.idx(session=1), ctx.idx(session=2), ctx.idx(session=3)
    setups = {"single (s1)": (s1, {"s2": s2, "s3": s3}), "multi (s1+s2)": (np.concatenate([s1, s2]), {"s3": s3})}
    for name, (train, tests) in setups.items():
        acc: Dict[str, List[float]] = {t: [] for t in tests}
        for rep in range(n_rep):
            perm = rng.permutation(ctx.n)
            n_enr = max(2, int(round(ctx.n * enrolled_frac)))
            enr, unk = perm[:n_enr], perm[n_enr:]
            if len(unk) == 0:
                continue
            remap = -np.ones(ctx.n, int)
            remap[enr] = np.arange(n_enr)
            y2 = remap[ctx.y]
            used = np.unique(np.concatenate([train] + list(tests.values())))
            X, Xf = prepare_inputs(ctx, "none", used)
            tr = train[y2[train] >= 0]
            scorer = tc.fit(X, Xf, y2, tr, {}, 0, n_enr)
            for t, tidx in tests.items():
                sc = scorer(tidx)
                yy = y2[tidx]
                g = sc[yy >= 0][np.arange((yy >= 0).sum()), yy[yy >= 0]]
                imp = sc[yy < 0].ravel()
                if len(g) and len(imp):
                    acc[t].append(verification_metrics(g, imp).eer)
        for t, v in acc.items():
            if v:
                rows.append({"scenario": "S9 open-set (unseen impostors)", "setting": f"{name} -> {t}", "group": name,
                             "model": tc.name, "test": t, "fuse_k": 1, "norm": "none", "bands": "all",
                             "eer": float(np.mean(v)), "eer_std": float(np.std(v)), "acc": float("nan"),
                             "acc_std": float("nan"), "n_test": int(len(tests[t])), "n_runs": len(v)})
                log(f"  S9 open-set | {name} -> {t}: EER {100 * np.mean(v):.1f}% ± {100 * np.std(v):.1f} over {len(v)} splits")
    return rows


def sc_ladder(ctx, M, P, seeds, log):
    """Step-by-step recipe, cumulative, always tested on session 3. Each step is pre-defined
    (not selected on the test data) so the table shows what each idea contributes."""
    s1, s2, s3 = ctx.idx(session=1), ctx.idx(session=2), ctx.idx(session=3)
    steps = [("0 baseline: enrol s1", s1, "none", 1),
             ("1 + per-session alignment (EA)", s1, "ea", 1),
             ("2 + enrol on s1+s2", np.concatenate([s1, s2]), "ea", 1),
             ("3 + fuse 5 clips per attempt", np.concatenate([s1, s2]), "ea", 5)]
    rows = []
    for label, train, mode, k in steps:
        rows += evaluate(ctx, M, train, {"s3": s3}, P, seeds, mode=mode, fuse=(k,), scenario="S10 improvement ladder",
                         setting=label, group=label, log=log)
    return rows


SCENARIOS = {"S1": sc_same_session, "S2": sc_cross_session, "S3": sc_multi_session, "S4": sc_enrol_size,
             "S5": sc_normalisation, "S6": sc_condition, "S7": sc_fusion, "S8": sc_bands,
             "S9": sc_openset, "S10": sc_ladder}
SCENARIO_TITLES = {"S1": "same-session", "S2": "cross-session, one enrolment session", "S3": "multi-session enrolment",
                   "S4": "how much enrolment data from the new session", "S5": "session normalisation",
                   "S6": "eyes-open vs eyes-closed", "S7": "multi-clip score fusion", "S8": "band-power feature set",
                   "S9": "open-set (never-seen impostors)", "S10": "improvement ladder"}


def run_all(ctx, *, scenarios=None, use_cnn=True, cnn_scenarios=("S1", "S2", "S3", "S10"), seeds=(0, 1, 2, 3, 4),
            max_epochs=60, out_dir=None, resume=True, log=print):
    """Run the chosen scenarios (resumable: finished scenarios are loaded from out_dir)."""
    scenarios = list(scenarios or SCENARIOS)
    fast = [LogRegBandPower(), RiemannianTangent(), TemplateCosine()]
    cnn = EEGNetStyleCNN(max_epochs=max_epochs)
    log("tuning on session-1 training data only:")
    P = tune_models(ctx, fast + ([cnn] if use_cnn else []), seed=seeds[0], log=log)
    out = Path(out_dir) if out_dir else None
    if out:
        (out / "partial").mkdir(parents=True, exist_ok=True)
    rows: List[Dict[str, Any]] = []
    for key in scenarios:
        f = out / "partial" / f"{key}.json" if out else None
        if resume and f is not None and f.exists():
            part = json.loads(f.read_text())
            log(f"{key} {SCENARIO_TITLES[key]}: loaded {len(part)} saved rows (delete {f.name} to re-run)")
        else:
            log(f"{key} {SCENARIO_TITLES[key]}")
            models = fast + ([cnn] if (use_cnn and key in cnn_scenarios) else [])
            part = SCENARIOS[key](ctx, models, P, list(seeds), log)
            if f is not None:
                f.write_text(json.dumps(part))
        rows += part
    return {"rows": rows, "params": P, "n_subjects": ctx.n, "seeds": list(seeds)}


# ------------------------------- reporting --------------------------------- #
def _fmt(m, s):
    return f"{100 * m:.1f}" if (s is None or not np.isfinite(s) or s == 0) else f"{100 * m:.1f} ± {100 * s:.1f}"


def _md(df, index=True):
    """DataFrame -> markdown table without needing the optional `tabulate` package."""
    d = df.reset_index() if index else df
    head = "| " + " | ".join(str(c) for c in d.columns) + " |"
    sep = "|" + "---|" * len(d.columns)
    body = ["| " + " | ".join("" if (isinstance(v, float) and v != v) else str(v) for v in r) + " |" for r in d.itertuples(index=False)]
    return "\n".join([head, sep] + body)


def make_report(res, out_dir, tag="run"):
    """results.csv/json, results.md (all tables) and figures. Returns markdown."""
    import pandas as pd
    import matplotlib.pyplot as plt

    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    df = pd.DataFrame(res["rows"])
    df.to_csv(out / "results.csv", index=False)
    (out / "results.json").write_text(json.dumps(res, indent=2))
    md = [f"# BED results ({res['n_subjects']} subjects, run '{tag}')\n",
          "EER in %, lower is better, chance is 50%. `±` = std over seeds/splits (deterministic models have none).\n",
          "Tuned on session-1 training data only: " + json.dumps(res["params"]) + "\n"]
    figs = []

    def table(key, title, index, columns, filt=None, note=""):
        d = df[df.scenario.str.startswith(key + " ")]
        if filt is not None:
            d = d[filt(d)]
        if d.empty:
            return None
        d = d.assign(v=[_fmt(a, b) for a, b in zip(d.eer, d.eer_std)])
        md.append(f"\n## {key} {title}\n")
        if note:
            md.append(note + "\n")
        if isinstance(columns, list):  # flatten (model, k) headers
            d = d.assign(col=d.model + " k=" + d.fuse_k.astype(str))
            pv = d.pivot_table(index=index, columns="col", values="v", aggfunc="first", sort=False)
        else:
            pv = d.pivot_table(index=index, columns=columns, values="v", aggfunc="first", sort=False)
        md.append(_md(pv) + "\n")
        return d

    d = df[df.scenario.str.startswith(("S1 ", "S2 "))]
    if not d.empty:
        g = d.groupby(["group", "model"], sort=False).eer.mean().unstack("model") * 100
        md.append("\n## S1+S2 Headline: same session vs 1-week vs 2-week gap (mean EER %)\n")
        md.append(_md(g.round(1)) + "\n")
        g.plot(kind="bar", figsize=(7, 4)); plt.ylabel("EER (%)"); plt.xticks(rotation=0)
        plt.title("EER grows with the time gap"); plt.axhline(50, ls=":", c="grey"); plt.tight_layout()
        plt.savefig(out / "fig1_time_gap.png", dpi=150); plt.close(); figs.append("fig1_time_gap.png")
        d2 = df[df.scenario.str.startswith("S2 ")].copy()
        md.append("\n### S2 every enrol -> test pair (EER %)\n")
        md.append(_md(d2.assign(v=[_fmt(a, b) for a, b in zip(d2.eer, d2.eer_std)]).pivot_table(
            index="setting", columns="model", values="v", aggfunc="first", sort=False)) + "\n")
    s3 = df[df.scenario.str.startswith("S3 ")]
    s2 = df[df.scenario.str.startswith("S2 ")]
    if not s3.empty and not s2.empty:
        s2t = s2.groupby(["test", "model"]).eer.mean().rename("single").reset_index()
        s3t = s3.groupby(["test", "model"]).eer.mean().rename("multi").reset_index()
        m = s2t.merge(s3t, on=["test", "model"])
        md.append("\n## S3 Single vs multi-session enrolment, same test session (mean EER %)\n")
        md.append("single = average over the two possible one-session enrolments.\n")
        md.append(_md(m.assign(single=(100 * m.single).round(1), multi=(100 * m.multi).round(1),
                               change=(100 * (m.multi - m.single)).round(1)), index=False) + "\n")
        pv = m.assign(label=m.model + " / " + m.test).set_index("label")[["single", "multi"]] * 100
        pv.plot(kind="bar", figsize=(8, 4)); plt.ylabel("EER (%)"); plt.xticks(rotation=60, ha="right", fontsize=7)
        plt.title("Enrolling on two sessions vs one"); plt.tight_layout()
        plt.savefig(out / "fig2_multisession.png", dpi=150); plt.close(); figs.append("fig2_multisession.png")
    d = table("S4", "Enrolment size: s1 + n clips per user from s2, tested on s3 (2 s per clip)", "setting", "model")
    if d is not None:
        for mname, gg in d.groupby("model"):
            plt.plot(gg.group.astype(int), 100 * gg.eer, marker="o", label=mname)
        plt.xlabel("clips per user from session 2"); plt.ylabel("EER on session 3 (%)"); plt.legend(fontsize=7)
        plt.title("A little data from the new session helps"); plt.tight_layout()
        plt.savefig(out / "fig3_enrol_size.png", dpi=150); plt.close(); figs.append("fig3_enrol_size.png")
    table("S5", "Session normalisation (" + "; ".join(f"{k}: {v}" for k, v in NORM_HELP.items()) + ")", ["setting", "test"], "model")
    table("S6", "Resting condition", ["setting", "test"], "model")
    d = table("S7", "Score fusion over k consecutive 2 s clips", ["setting", "test"], ["model", "fuse_k"], note="k = clips averaged per login attempt")
    if d is not None:
        for (mname, st), gg in d[d.test == "s3"].groupby(["model", "setting"]):
            plt.plot(gg.fuse_k, 100 * gg.eer, marker="o", label=f"{mname} | {st}")
        plt.xlabel("clips fused per attempt"); plt.ylabel("EER on session 3 (%)"); plt.legend(fontsize=6)
        plt.title("Longer login attempts"); plt.tight_layout()
        plt.savefig(out / "fig4_fusion.png", dpi=150); plt.close(); figs.append("fig4_fusion.png")
    table("S8", "Band-power feature set (LogReg)", ["setting", "test"], "model")
    table("S9", "Open-set: impostors are people never enrolled", "setting", "model")
    d = table("S10", "Improvement ladder, tested on session 3 (cumulative, steps fixed in advance)", "setting", "model")
    if d is not None:
        pv = d.pivot_table(index="setting", columns="model", values="eer", sort=False) * 100
        pv.plot(kind="bar", figsize=(8, 4)); plt.ylabel("EER on session 3 (%)"); plt.xticks(rotation=20, ha="right", fontsize=7)
        plt.title("Improvement ladder"); plt.legend(fontsize=7); plt.tight_layout()
        plt.savefig(out / "fig5_ladder.png", dpi=150); plt.close(); figs.append("fig5_ladder.png")
    md.append("\n## Figures\n" + "\n".join(f"- {f}" for f in figs) + "\n")
    text = "\n".join(md)
    (out / "results.md").write_text(text, encoding="utf-8")
    return text, figs


def log_run(work_dir, tag, config, res):
    """Append one line per run so the history of experiments is never lost."""
    p = Path(work_dir) / "runs"
    p.mkdir(parents=True, exist_ok=True)
    rec = {"tag": tag, "time": time.strftime("%Y-%m-%d %H:%M:%S"), "config": config,
           "n_subjects": res["n_subjects"], "n_rows": len(res["rows"]), "params": res["params"]}
    with open(p / "runs.jsonl", "a") as f:
        f.write(json.dumps(rec) + "\n")

In [ ]:
# 6. Load the BED module written above (cell 5) and check it imports
import importlib
from eegauth import bed
importlib.reload(bed)
print('BED module ready; channels:', bed.BED_CHANNELS)

In [ ]:
# 7. Find your BED files in Google Drive (read in place; a zip is extracted from a copy on Colab's disk)
import glob, re, shutil, zipfile
from pathlib import Path

LOCAL_RAW = '/content/bed_raw'
os.makedirs(LOCAL_RAW, exist_ok=True)
os.makedirs(WORK, exist_ok=True)
DRIVE_ROOT = '/content/drive/MyDrive'

def has_bed(fns):
    return (any(re.fullmatch(r's\d+_s\d+\.mat', f, re.I) for f in fns) or
            any(f.lower().endswith('.zip') and ('bed' in f.lower() or 'parsed' in f.lower()) for f in fns))

def search_drive(max_depth=6):
    hits = []
    base = DRIVE_ROOT.count('/')
    for dp, dns, fns in os.walk(DRIVE_ROOT):
        dns[:] = [d for d in dns if d not in ('.Trash', '.shortcut-targets-by-id') and os.path.join(dp, d) != WORK]
        if dp.count('/') - base >= max_depth:
            dns[:] = []
        if has_bed(fns):
            hits.append(dp)
    return hits

if os.path.isdir(DRIVE_BED_DIR) and (bed.find_mats(DRIVE_BED_DIR) or glob.glob(f'{DRIVE_BED_DIR}/**/*.zip', recursive=True)):
    roots = [DRIVE_BED_DIR]
else:
    print(f'{DRIVE_BED_DIR} not found or has no BED files; searching your Drive ...')
    roots = search_drive()
    print('found BED files in:', roots)
assert roots, ('No BED files (sN_sM.mat or a zip) found in Drive. Check that the upload finished, then set '
               'DRIVE_BED_DIR in cell 4. Top-level Drive folders: ' + str(sorted(os.listdir(DRIVE_ROOT))[:30]))

if not bed.find_mats(LOCAL_RAW):                      # unpack zips once per session (originals stay in Drive)
    for root in roots:
        for z in glob.glob(f'{root}/**/*.zip', recursive=True):
            tmp = '/content/_bed_tmp.zip'
            shutil.copy(z, tmp)
            try:
                with zipfile.ZipFile(tmp) as zf:
                    members = [m for m in zf.namelist() if m.lower().endswith('.mat')]
                    zf.extractall(LOCAL_RAW, members)
                    print(f'extracted {len(members)} .mat files from {Path(z).name}')
            except zipfile.BadZipFile:
                print('skipping bad zip:', z)
            finally:
                os.remove(tmp)

mats = {}
for r in roots + [LOCAL_RAW]:
    for k, v in bed.find_mats(r).items():
        mats.setdefault(k, v)
print(len(mats), 'recordings found | subjects:', sorted({s for s, _ in mats}), '| sessions:', sorted({m for _, m in mats}))
assert mats, 'No files named s<N>_s<M>.mat found in Drive (or in a zip). Upload the RAW_PARSED files.'

In [ ]:
# 8. Inspect ONE raw file so we use the real format (shapes, rate, channel columns, event labels)
import pprint
first = next(iter(sorted(mats.items())))[1]
info = bed.inspect_mat(first)
print('file:', first.name)
print('recording shape (time x columns):', info['recording_shape'])
print('documented columns:', info['assumed_columns'])
print('documented rate:', info['assumed_sampling_rate_hz'], 'Hz | measured from timestamps:', info['measured_rate_hz'])
print('timestamp range:', info['timestamp_range'], '| NaNs in recording:', info['nan_in_recording'])
print('events:', info['n_events'], '| keys in ADDITIONAL_INFO:', info['info_keys'])
print('\nfirst events:')
pprint.pprint(info['first_events'], width=120)
print('\nevent labels (count) and how REST_RULES classify them:')
for lab, n in sorted(info['label_counts'].items(), key=lambda kv: -kv[1]):
    print(f'  {n:4d}  {lab[:80]!r:85} -> {bed.classify_rest(lab, REST_RULES)}')
assert info['recording_shape'][1] == 17, 'unexpected column count: update bed.EMOTIV_COLUMNS from the output above'
assert any(bed.classify_rest(l, REST_RULES) for l in info['label_counts']), (
    'No event matched the resting-state rules. Read the labels above and edit REST_RULES / LABEL_FIELD in cell 4.')

In [ ]:
# 9. Preprocess per subject (resumable; saved to Drive after every subject): 1-40 Hz band-pass -> rest segments -> 160 Hz -> 2 s clips
meta = bed.prepare_dataset(None, WORK, mats=mats, n_subjects=N_SUBJECTS, rules=REST_RULES, label_field=LABEL_FIELD,
                           target_fs=TARGET_FS, l_freq=BAND[0], h_freq=BAND[1], clip_s=CLIP_S, max_seconds=MAX_SECONDS)
print('done')

In [ ]:
# 10. Dataset summary
size_mb = os.path.getsize(f'{WORK}/bed_subset.npz') / 1e6
print(f'bed_subset.npz: {size_mb:.1f} MB | clips: {meta["n_clips"]} | subjects: {len(meta["subjects"])}')
print('clips per session:', meta['clips_per_session'], '| per task:', meta['clips_per_task'])
print('skipped recordings:', len(meta['skipped']))
for s in meta['skipped'][:20]:
    print('  ', s)

In [ ]:
# 11. Build the study set: only subjects that have all 3 sessions
import numpy as np
with np.load(f'{WORK}/bed_subset.npz') as z:
    data = {k: z[k] for k in z.files}
ctx = bed.Ctx(data, sfreq=TARGET_FS, train_frac=TRAIN_FRAC, val_frac=VAL_FRAC)
OUT = f'{WORK}/results_{RUN_TAG}'
os.makedirs(OUT, exist_ok=True)
print('results folder:', OUT)

In [ ]:
# 12. Run all scenarios (tuning first, then S1..S10; each finished scenario is saved, so you can resume)
import time
t0 = time.time()
res = bed.run_all(ctx, scenarios=SCENARIOS, use_cnn=USE_CNN, cnn_scenarios=CNN_SCENARIOS,
                  seeds=list(range(N_SEEDS)), max_epochs=MAX_EPOCHS, out_dir=OUT, resume=RESUME)
print(f'done in {(time.time() - t0) / 60:.1f} min, {len(res["rows"])} result rows')

In [ ]:
# 13. Tables + figures, and log this run in the history file
config = dict(N_SUBJECTS=N_SUBJECTS, N_SEEDS=N_SEEDS, BAND=BAND, CLIP_S=CLIP_S, TARGET_FS=TARGET_FS,
              TRAIN_FRAC=TRAIN_FRAC, VAL_FRAC=VAL_FRAC, MAX_EPOCHS=MAX_EPOCHS, SCENARIOS=SCENARIOS,
              USE_CNN=USE_CNN, REST_RULES=REST_RULES)
text, figs = bed.make_report(res, OUT, tag=RUN_TAG)
bed.log_run(WORK, RUN_TAG, config, res)
from IPython.display import Image, Markdown, display
display(Markdown(text))
for f in figs:
    display(Image(f'{OUT}/{f}'))

In [ ]:
# 14. Compare with earlier runs (history): every run logged so far
import json
for line in open(f'{WORK}/runs/runs.jsonl'):
    r = json.loads(line)
    print(r['time'], '|', r['tag'], '|', r['n_subjects'], 'subjects |', r['n_rows'], 'rows | scenarios', r['config'].get('SCENARIOS'))

In [ ]:
# 15. Optional: download results (tables, csv, json, figures; no EEG data) as one zip
import shutil
from google.colab import files
shutil.make_archive('/content/bed_results', 'zip', OUT)
files.download('/content/bed_results.zip')
# MyDrive/eeg_auth_bed/bed_subset.npz and processed/ are restricted data derived from BED: keep that folder private.